# Task 6: House Price Prediction
**DevelopersHub Corporation — AI/ML Engineering Internship**

---

## Problem Statement
Predict house prices based on property features such as size, location, number of rooms, and other structural attributes.

## Goal
- Preprocess and explore the housing dataset
- Train and compare regression models (Linear Regression & Gradient Boosting)
- Visualize predicted vs actual prices
- Evaluate using MAE and RMSE

## Dataset
The **California Housing Dataset** (from sklearn) contains data on 20,640 California census blocks with 8 features.

| Feature | Description |
|---|---|
| MedInc | Median income in block (tens of thousands USD) |
| HouseAge | Median house age in block |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block population |
| AveOccup | Average household occupancy |
| Latitude | Block latitude |
| Longitude | Block longitude |
| **MedHouseVal** | **Target: Median house value (hundreds of thousands USD)** |

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('plots', exist_ok=True)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries loaded successfully!')

## Step 2: Load the Dataset

In [ ]:
# Load California Housing dataset (built into sklearn — no download needed)
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Step 3: Data Inspection & Preprocessing

In [ ]:
# Preview
df.head()

In [ ]:
# Data types and nulls
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Remove extreme outliers in AveRooms and AveOccup (unrealistic values)
print(f'Shape before outlier removal: {df.shape}')
df = df[df['AveRooms'] < 50]      # Remove blocks with avg >50 rooms
df = df[df['AveOccup'] < 20]      # Remove blocks with avg >20 occupants
df = df[df['AveBedrms'] < 10]     # Remove blocks with avg >10 bedrooms
print(f'Shape after outlier removal:  {df.shape}')

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
plt.figure(figsize=(9, 5))
sns.histplot(df['MedHouseVal'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Median House Values', fontsize=14, fontweight='bold')
plt.xlabel('Median House Value (hundreds of thousands USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('plots/target_distribution.png', dpi=150)
plt.show()

In [ ]:
# Feature distributions
features = ['MedInc', 'HouseAge', 'AveRooms', 'Population', 'AveOccup']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), features):
    sns.histplot(df[feat], bins=40, ax=ax, kde=True, color='teal')
    ax.set_title(feat)
    ax.set_xlabel('')

axes[1, 2].set_visible(False)
plt.tight_layout()
plt.savefig('plots/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', square=True, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Income vs House Value (strongest relationship)
plt.figure(figsize=(9, 6))
plt.scatter(df['MedInc'], df['MedHouseVal'], alpha=0.2, s=10, color='steelblue')
plt.title('Median Income vs House Value', fontsize=14, fontweight='bold')
plt.xlabel('Median Income (tens of thousands USD)')
plt.ylabel('Median House Value (hundreds of thousands USD)')
plt.tight_layout()
plt.savefig('plots/income_vs_price.png', dpi=150)
plt.show()

In [ ]:
# Geographic distribution of house prices
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'], cmap='YlOrRd',
    s=5, alpha=0.5
)
plt.colorbar(scatter, label='Median House Value')
plt.title('Geographic Distribution of House Prices in California', fontsize=13, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.savefig('plots/geo_distribution.png', dpi=150)
plt.show()

> **Insight:** Coastal areas (Los Angeles, San Francisco Bay) have significantly higher house values than inland regions — location is a major price driver.

## Step 5: Model Training

In [ ]:
# Define features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'Features:         {X_train.shape[1]}')

In [ ]:
# --- Model 1: Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)

lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2   = r2_score(y_test, lr_preds)

print('=== Linear Regression ===')
print(f'MAE:  {lr_mae:.4f}')
print(f'RMSE: {lr_rmse:.4f}')
print(f'R²:   {lr_r2:.4f}')

In [ ]:
# --- Model 2: Gradient Boosting Regressor ---
gb = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.1,
    max_depth=4, random_state=42
)
gb.fit(X_train, y_train)  # Tree-based: no scaling needed
gb_preds = gb.predict(X_test)

gb_mae  = mean_absolute_error(y_test, gb_preds)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_r2   = r2_score(y_test, gb_preds)

print('=== Gradient Boosting Regressor ===')
print(f'MAE:  {gb_mae:.4f}')
print(f'RMSE: {gb_rmse:.4f}')
print(f'R²:   {gb_r2:.4f}')

## Step 6: Model Evaluation & Visualization

In [ ]:
# Model comparison bar chart
metrics_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Gradient Boosting'],
    'MAE':   [lr_mae, gb_mae],
    'RMSE':  [lr_rmse, gb_rmse],
    'R²':    [lr_r2, gb_r2]
})

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    bars = ax.bar(metrics_df['Model'], metrics_df[metric], color=['steelblue', 'darkorange'])
    ax.bar_label(bars, fmt='%.3f', fontsize=10)
    ax.set_title(metric, fontsize=12)
    ax.set_ylabel(metric)
    ax.set_xticklabels(metrics_df['Model'], rotation=10, ha='right')

plt.tight_layout()
plt.savefig('plots/model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Actual vs Predicted — both models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Actual vs Predicted House Prices', fontsize=14, fontweight='bold')

for ax, preds, title, color in [
    (axes[0], lr_preds, 'Linear Regression', 'steelblue'),
    (axes[1], gb_preds, 'Gradient Boosting', 'darkorange')
]:
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=color)
    ax.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        'r--', lw=2, label='Perfect Prediction'
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Price')
    ax.set_ylabel('Predicted Price')
    ax.legend()

plt.tight_layout()
plt.savefig('plots/actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance — Gradient Boosting
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': gb.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='darkorange')
plt.title('Feature Importance — Gradient Boosting', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Residual plot — Gradient Boosting
residuals = y_test - gb_preds

plt.figure(figsize=(9, 5))
plt.scatter(gb_preds, residuals, alpha=0.3, s=10, color='teal')
plt.axhline(y=0, color='red', linestyle='--', lw=2)
plt.title('Residual Plot — Gradient Boosting', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.tight_layout()
plt.savefig('plots/residuals.png', dpi=150)
plt.show()

## Summary & Final Insights

| Metric | Linear Regression | Gradient Boosting |
|---|---|---|
| MAE | ~0.53 | ~0.37 |
| RMSE | ~0.73 | ~0.51 |
| R² | ~0.60 | ~0.80 |

*(Exact values printed in output above)*

### Key Findings
- **Gradient Boosting significantly outperforms Linear Regression** — it captures non-linear relationships between features and price.
- **Median Income (`MedInc`)** is by far the strongest predictor of house prices — wealthier neighborhoods command much higher prices.
- **Location (Latitude/Longitude)** is the second most important factor — coastal proximity drives prices up substantially.
- **Average Rooms and House Age** have moderate influence, while **Population and AveOccup** contribute less.
- **Linear Regression** underfits the data (R² ≈ 0.60) — house pricing is inherently non-linear.
- **Residuals** are roughly centered around zero for Gradient Boosting, indicating a well-calibrated model with no systematic bias.

### Business Takeaway
For real-world house price prediction, ensemble methods like Gradient Boosting (or XGBoost/LightGBM) should be preferred over simple linear models. Income levels and geographic location are the two most critical factors in determining property value.